# H1 — Power Diagnostics (run before believing any negative result)

*(Formerly D1. Renamed because Appendix D of the report is the deployment
and decision-problem material — D.2 three decision problems, D.3 hubness, D.4
the allegory — and a notebook called D1 invited confusion with it. H is
reserved for tooling that checks the measurements rather than producing
them.)*

Implements the project's rule of thumb as executable checks, in triage
order:

| # | Check | Catches | Cost |
|---|---|---|---|
| 1 | **Rows per input dimension** | starved fits (systematically pessimistic R²) | arithmetic |
| 2 | **Alignment (shuffle test)** | silent row misalignment — the classic pipeline bug | seconds |
| 3 | **Artifact sanity** | NaN/Inf, zero vectors, duplicates, broken normalization | seconds |
| 4 | **Target geometry** | a collapsed or hubbed target — the most likely cause of a WRONG negative | seconds |
| 5 | **Cache-key integrity** | a checkpoint whose contents do not match the tag in its filename | seconds |

Runs on whatever artifacts exist in `DATA_DIR` — `activations.npz`
(Experiment A), `pairs.npz` (Experiment B), and any `e1_img_ckpt_*` /
`crossmodal_pairs*` caches from the E and G series — and prints
PASS / FAIL per check with the measured numbers.

**Check #4 is the one added last, and it matters most.** Section C.11
found a case where retrieval had collapsed to R@1 0.130 while the
mapping was fine: the target space had effective rank 6.5 of 768, and a
label-free whitening recovered R@1 to 0.478. A negative result measured
against a collapsed target is not evidence of absent structure — it is
evidence of an unreadable coordinate frame, and this check flags that
signature before it is misread.

**The logic of the shuffle test:** fit the same ridge map twice — once on
the data as-is, once with the target rows deliberately shuffled. If the
rows are truly aligned, the honest fit must beat the shuffled one
decisively (shuffled ≈ 0 or negative). If the two scores are close, the
"aligned" data was never aligned, and every downstream number is
meaningless. This doubles as a starvation detector: a strongly **negative**
shuffled score is the memorized-then-shrunk signature (run 1's −0.773).

In [ ]:
# =====================================================================
# WHERE THIS RUNS AND WHERE DATA GOES - read before running
# =====================================================================
# RUNTIME (choose in the Colab menu, or just open this locally):
#   Colab  : Runtime -> Change runtime type -> T4 / L4 GPU. A free T4 is
#            enough for most notebooks here; CPU works but is slow.
#   Local  : open in Jupyter on a machine with a CUDA GPU or Apple
#            Silicon. Nothing needs changing - the google.colab import
#            below fails harmlessly and it falls through to local mode.
#            To drive a local kernel from the Colab UI: install
#            jupyter_http_over_ws, launch jupyter with
#            --NotebookApp.allow_origin='https://colab.research.google.com'
#            and paste the printed localhost URL (with its token) into
#            Connect -> Connect to a local runtime.
#
# STORAGE (set STORAGE below):
#   "drive" : Google Drive at MyDrive/convergence_experiment  [DEFAULT]
#             Survives session timeouts, so checkpoints resume. On a
#             local machine this falls back to LOCAL_DIR automatically.
#   "local" : LOCAL_DIR on whatever machine is running. On a hosted
#             Colab VM this disk is ERASED at session end - downloads
#             and checkpoints do not survive.
#   "env"   : whatever DATA_DIR is already set to in the environment.
#
# The same folder can be shared between Colab and a local machine (the
# caches are plain .npz) - point both at one synced Drive folder.
# =====================================================================
import os
from pathlib import Path

STORAGE   = "drive"                     # "drive" | "local" | "env"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"

try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR is unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    if STORAGE == "drive":
        print("not on Colab - Drive unavailable, using LOCAL_DIR instead")
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())

DATA_DIR = Path(os.environ["DATA_DIR"])
DATA_DIR.mkdir(parents=True, exist_ok=True)

try:
    import torch
    _dev = ("cuda" if torch.cuda.is_available()
            else "mps" if getattr(torch.backends, "mps", None)
            and torch.backends.mps.is_available() else "cpu")
    _name = torch.cuda.get_device_name(0) if _dev == "cuda" else _dev
except ImportError:
    _dev, _name = "?", "torch not installed yet - run the pip cell"
print(f"environment : {'Colab' if IN_COLAB else 'local machine'}")
print(f"device      : {_dev} ({_name})")
print(f"DATA_DIR    : {DATA_DIR}")
if _dev == "cpu":
    print("WARNING: no GPU detected - encoding will take hours, not "
          "minutes")

In [ ]:
import numpy as np
from pathlib import Path
DATA_DIR = Path(os.environ['DATA_DIR'])
rng = np.random.default_rng(0)
RESULTS = []

def verdict(name, ok, detail):
    RESULTS.append((name, ok))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: {detail}")

def ridge(X, Y, a=1e-2):
    d = X.shape[1]
    return np.linalg.solve(X.T @ X + a * np.eye(d), X.T @ Y)

def heldout_r2(X, Y, frac=0.75, a=1e-2):
    n = len(X)
    idx = rng.permutation(n)
    k = int(n * frac)
    tr, te = idx[:k], idx[k:]
    W = ridge(X[tr], Y[tr], a)
    P = X[te] @ W
    ss = ((Y[te] - P) ** 2).sum()
    st = ((Y[te] - Y[te].mean(0)) ** 2).sum()
    return 1 - ss / st

def check_power(X, Y, label):
    """#1: rows per input dimension (governing ratio)."""
    rows = int(len(X) * 0.75)
    spd = rows / X.shape[1]
    verdict(f"{label} · rows/input-dim", spd >= 5,
            f"{rows}/{X.shape[1]} = {spd:.1f} "
            f"(>=5 required, >=10 comfortable)")
    return spd

def check_alignment(X, Y, label):
    """#2: aligned fit must decisively beat a shuffled fit."""
    r_true = heldout_r2(X, Y)
    Ys = Y[rng.permutation(len(Y))]
    r_shuf = heldout_r2(X, Ys)
    ok = (r_true - r_shuf) > 0.2 and r_true > 0
    verdict(f"{label} · alignment (shuffle test)", ok,
            f"aligned R2={r_true:.3f} vs shuffled R2={r_shuf:.3f} "
            f"(gap {r_true - r_shuf:.3f}; need >0.2)")
    if r_shuf < -0.3:
        print(f"      note: strongly negative shuffled R2 "
              f"({r_shuf:.3f}) = memorize-then-shrink signature; "
              f"if the ALIGNED score is also low, suspect starvation "
              f"before absence of structure")

def check_artifact(M, label, expect_unit=False):
    """#3: NaN/Inf, zero rows, duplicates, normalization."""
    flat = M.reshape(-1, M.shape[-1])
    bad = int(np.isnan(flat).any(1).sum() + np.isinf(flat).any(1).sum())
    verdict(f"{label} · finite values", bad == 0,
            f"{bad} rows with NaN/Inf")
    norms = np.linalg.norm(flat, axis=1)
    nz = int((norms < 1e-6).sum())
    verdict(f"{label} · non-zero vectors", nz == 0, f"{nz} zero rows")
    sample = flat[rng.choice(len(flat), min(4000, len(flat)),
                             replace=False)]
    _, counts = np.unique(np.round(sample, 5), axis=0,
                          return_counts=True)
    dups = int((counts > 1).sum())
    verdict(f"{label} · duplicate rows", dups <= len(sample) * 0.01,
            f"{dups} duplicated rows in a {len(sample)}-row sample")
    if expect_unit:
        dev = float(np.abs(norms - 1).max())
        verdict(f"{label} · L2-normalized", dev < 1e-3,
                f"max |norm-1| = {dev:.2e}")


# ---------------------------------------------------------------- #4
def check_geometry(Y, label, k=10):
    """Is the TARGET space readable by cosine at all?

    This is the check that Section C.11 exists because of. A target with
    a collapsed spectrum produces poor retrieval even when the mapping
    is perfect - and the fix is a free linear transform, not more data.
    Reading a negative result from such a space as 'no shared structure'
    is the single most likely misinterpretation in this project.
    """
    from scipy.stats import skew
    Yc = Y - Y.mean(0)
    sv = np.linalg.svd(Yc, full_matrices=False, compute_uv=False)
    pr = sv ** 2 / (sv ** 2).sum(); pr = pr[pr > 0]
    er = float(np.exp(-(pr * np.log(pr)).sum()))
    frac = er / Y.shape[1]

    Yn = Y / (np.linalg.norm(Y, axis=1, keepdims=True) + 1e-12)
    i = rng.integers(0, len(Y), 3000); j = rng.integers(0, len(Y), 3000)
    m = i != j
    pc = float((Yn[i[m]] * Yn[j[m]]).sum(1).mean())

    sub = Yn[rng.permutation(len(Yn))[:min(2000, len(Yn))]]
    S = sub @ sub.T; np.fill_diagonal(S, -9)
    nn = np.argsort(-S, 1)[:, :k]
    hb = float(skew(np.bincount(nn.ravel(), minlength=len(sub))))

    ok = (frac >= 0.02) and (pc < 0.90)
    verdict(f"{label} · target geometry", ok,
            f"eff.rank {er:.1f}/{Y.shape[1]} ({100*frac:.1f}%), "
            f"pair-cos {pc:+.3f}, N{k} skew {hb:.2f}")
    if not ok:
        print("       ^ COLLAPSED TARGET. Cosine-based scores from this "
              "space will\n"
              "         understate the correspondence. Before reporting a "
              "negative,\n"
              "         refit against a WHITENED target (fit the whitening "
              "on train\n"
              "         rows only) - see Section C.11. A whitened retry "
              "recovered\n"
              "         R@1 0.130 -> 0.478 on exactly this signature.")
    if hb > 3.0:
        print(f"       ^ HUBBED (skew {hb:.2f}): a few items are the "
              f"nearest neighbour of\n"
              "         disproportionately many queries, which depresses "
              "R@1 while\n"
              "         leaving verification AUC intact - see Section D.3.")
    return er, pc, hb


# ---------------------------------------------------------------- #5
def check_cache_key(path, label):
    """Does a checkpoint's CONTENT match the tag in its filename?

    Every silent-corruption bug in this project came from a cache whose
    name promised one thing and whose contents were another: a resumed
    run under a different encoder, a stale teacher, a reference arm from
    the wrong configuration.
    """
    import re
    name = Path(path).name
    d = np.load(str(path))
    key = "img" if "img" in d.files else d.files[0]
    W = d[key].shape[1]
    exp = {"dinov2-small": (384, 768), "dinov2-base": (768, 1536),
           "dinov2-large": (1024, 2048)}
    hit = [(t, w) for t, w in exp.items() if t in name]
    if not hit:
        verdict(f"{label} · cache key", True,
                f"no size tag in filename to verify (width {W})")
        return
    tag, widths = hit[0]
    ok = W in widths
    verdict(f"{label} · cache key", ok,
            f"filename says {tag}, content width {W} "
            f"(expected {widths[0]} cls or {widths[1]} cls+patch)")
    if not ok:
        print("       ^ NAME/CONTENT MISMATCH - this cache was written by "
              "a different\n         run than its filename claims. "
              "Delete it and re-extract.")

In [ ]:
# ---- Experiment A artifacts ----
f = DATA_DIR / 'activations.npz'
if f.exists():
    d = np.load(f)
    A, B = d['A_layers'], d['B_layers']
    print(f"activations.npz: A {A.shape}, B {B.shape}\n")
    L = A.shape[0] // 2                      # a middle layer
    X, Y = A[L].astype(np.float64), B[L].astype(np.float64)
    check_power(X, Y, f"ExpA L{L}")
    check_alignment(X, Y, f"ExpA L{L}")
    check_artifact(A, "ExpA A_layers")
    check_artifact(B, "ExpA B_layers")
    if 'R_layers' in d:
        check_artifact(d['R_layers'], "ExpA R_layers (control)")
else:
    print("activations.npz not found - skipping Experiment A checks")

In [ ]:
# ---- Experiment B artifacts ----
f = DATA_DIR / 'pairs.npz'
if f.exists():
    d = np.load(f)
    mob, sig = d['mob_img'].astype(np.float64), d['sig_img'].astype(np.float64)
    print(f"pairs.npz: mob {mob.shape}, sig {sig.shape}\n")
    check_power(mob, sig, "ExpB mob->sig")
    check_alignment(mob, sig, "ExpB mob->sig")
    check_artifact(mob, "ExpB mob_img", expect_unit=True)
    check_artifact(sig, "ExpB sig_img", expect_unit=True)
    if 'sig_txt' in d:
        check_artifact(d['sig_txt'], "ExpB sig_txt", expect_unit=True)
    check_geometry(sig, "ExpB sig_img (target)")
else:
    print("pairs.npz not found - skipping Experiment B checks")

# ---- E/G-series artifacts: cached encoder outputs and pair files ----
for f in sorted(DATA_DIR.glob("e1_img_ckpt_*.npz")):
    check_cache_key(f, f"cache {f.stem[-24:]}")
for f in sorted(DATA_DIR.glob("crossmodal_pairs*.npz")):
    d = np.load(str(f))
    if "txt" not in d.files:
        continue
    lab = f.stem[-28:]
    Yt = d["txt"].astype(np.float64)
    check_artifact(Yt, f"{lab} txt")
    check_geometry(Yt, f"{lab} txt (target)")
    if "img" in d.files:
        Xi = d["img"].astype(np.float64)
        check_power(Xi, Yt, lab)
        check_alignment(Xi, Yt, lab)

print("\n" + "=" * 56)
fails = [n for n, ok in RESULTS if not ok]
if not RESULTS:
    print("NO ARTIFACTS FOUND - nothing checked")
elif not fails:
    print(f"ALL {len(RESULTS)} CHECKS PASS - negative results from these")
    print("artifacts can be interpreted as real absences of structure")
else:
    print(f"{len(fails)} CHECK(S) FAILED - do NOT interpret any negative")
    print("result until these are resolved:")
    for n in fails:
        print("  -", n)

## When to run this

1. **After every extraction** (a fresh A1 or B1 run) — before A2/A3/B2
   consume the artifacts. Catches alignment and sanity problems at the
   source instead of as confusing results two stages later.
2. **Before believing any negative or below-threshold result** — this is
   the primary purpose. Run-1's failure would have been caught here in
   seconds: rows/dim = 1.6 fails check #1, and the control's −0.773
   pattern is flagged by the shuffle test's note.
3. **After any pipeline change** — new dataset, new preprocessing, a
   model swap (e.g. MobileCLIP2 refit), fp16 storage, different pooling.
   Anything that touches how the matrices are produced.
4. **Not needed** after runs whose results you accept and whose pipeline
   is unchanged — the checks certify the measurement, not the science.

What it deliberately does not cover: check #3 of the rule of thumb
(pooling/preprocessing *choices*) can only be partially automated — this
notebook catches broken outputs (NaN, zero rows, un-normalized vectors),
but whether the preprocessing is the *right* one (identity normalization
on iOS, correct layer choice) remains a design review, not a script.